In [ ]:
import os
import warnings
import logging
import cv2
import numpy as np
import pandas as pd
import glob
from ultralytics import YOLO
from hmmlearn import hmm

# Silence warnings to keep the console clean
warnings.filterwarnings("ignore")
logging.getLogger("hmmlearn").setLevel(logging.ERROR)
os.environ["YOLO_VERBOSE"] = "False"

# --- CONFIGURATION ---
MODEL_PATH = '../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt'
VIDEO_FOLDER = '../yolov8_model/videos'
OUTPUT_FILE = 'final_match_scores.csv'
FACE_CAPACITIES = [3, 1, 2, 2, 1, 3]

def build_face_hmm(capacity):
    num_states = capacity + 1
    model = hmm.MultinomialHMM(n_components=num_states, n_trials=3, n_iter=100)
    model.startprob_ = np.zeros(num_states); model.startprob_[0] = 1.0
    trans_mtx = np.zeros((num_states, num_states))
    for i in range(num_states):
        for j in range(num_states):
            if i == j: trans_mtx[i][j] = 0.90
            elif j == i + 1: trans_mtx[i][j] = 0.09
            elif j < i: trans_mtx[i][j] = 0.01
        trans_mtx[i] /= trans_mtx[i].sum()
    model.transmat_ = trans_mtx
    emit_mtx = np.full((num_states, 4), 0.05)
    for i in range(num_states): emit_mtx[i][min(i, 3)] = 0.85
    for i in range(num_states): emit_mtx[i] /= emit_mtx[i].sum()
    model.emissionprob_ = emit_mtx
    return model

model = YOLO(MODEL_PATH)
all_final_data = []

# Get all video files
video_files = glob.glob(os.path.join(VIDEO_FOLDER, "*.mp4"))

for video_path in video_files:
    video_name = os.path.basename(video_path)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    stop_frame = int(2.38 * 60 * fps)
    
    # These reset for EVERY video
    reef_obs_logs = {} 
    last_known_reefs = {} 
    reef_first_seen = {}
    frame_count = 0

    print(f"Processing: {video_name}...")

    while cap.isOpened():
        if frame_count > stop_frame: break
        success, frame = cap.read()
        if not success: break

        h, w, _ = frame.shape
        y_offset = int(h * (3/5))
        cropped_frame = frame[y_offset:h, 0:w]
        
        results = model.track(cropped_frame, persist=True, conf=0.3, verbose=False)
        frame_detections = []

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.tolist()
            ids = results[0].boxes.id.int().tolist()
            clss = results[0].boxes.cls.int().tolist()

            for box, obj_id, cls in zip(boxes, ids, clss):
                label = model.names[cls]
                if label == 'reef':
                    last_known_reefs[obj_id] = box
                    if obj_id not in reef_first_seen:
                        reef_first_seen[obj_id] = frame_count
                else:
                    frame_detections.append({'label': label, 'coords': box})

        for r_id, r_box in last_known_reefs.items():
            if r_id not in reef_obs_logs:
                reef_obs_logs[r_id] = {i: [] for i in range(6)}

            rx1, ry1, rx2, ry2 = r_box
            reef_w, reef_h = rx2 - rx1, ry2 - ry1
            l4_top, l4_bottom = ry1 - (reef_h * 0.05), ry1 + (reef_h * 0.30)
            face_width = reef_w / 6
            
            for face_idx in range(6):
                fx1 = rx1 + (face_idx * face_width)
                fx2 = fx1 + face_width
                count = 0
                for det in frame_detections:
                    dx1, dy1, dx2, dy2 = det['coords']
                    mx, my = (dx1 + dx2) / 2, (dy1 + dy2) / 2
                    if fx1 <= mx <= fx2 and l4_top <= my <= l4_bottom:
                        if det['label'] == 'coral': 
                            count += 1
                
                reef_obs_logs[r_id][face_idx].append(min(count, 3))

        frame_count += 1
    cap.release()

    # --- HMM Decoding (Now strictly for the top 2 reefs found) ---
    # We sort by length of observation to make sure we get the "real" reefs
    sorted_ids = sorted(reef_obs_logs.keys(), key=lambda x: len(reef_obs_logs[x][0]), reverse=True)[:2]

    for r_id in sorted_ids:
        faces = reef_obs_logs[r_id]
        reef_score = 0
        
        # Calculate Timestamp
        ts_sec = reef_first_seen[r_id] / fps
        timestamp = f"{int(ts_sec//60):02d}:{int(ts_sec%60):02d}"

        for f_idx in range(6):
            obs = faces[f_idx]
            if not obs: continue
            
            hmm_model = build_face_hmm(FACE_CAPACITIES[f_idx])
            obs_one_hot = np.zeros((len(obs), 4), dtype=int)
            for i, v in enumerate(obs):
                obs_one_hot[i, min(int(v), 3)] = 1
            
            _, states = hmm_model.decode(obs_one_hot, algorithm="viterbi")
            reef_score += states[-1]

        all_final_data.append({
            'video': video_name,
            'reef_id': r_id,
            'first_seen': timestamp,
            'l4_total': reef_score
        })
        print(f"  - Reef {r_id} (seen at {timestamp}): {reef_score} points")

# Save everything to one master file
pd.DataFrame(all_final_data).to_csv(OUTPUT_FILE, index=False)
print(f"\nDone! Results for all videos saved to {OUTPUT_FILE}")

Processing: Qualification 72 - 2025 Iowa Regional.mp4...
  - Reef 14 (seen at 00:03): 0 points
  - Reef 27 (seen at 00:05): 0 points
Processing: Qualification 79 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4...
  - Reef 934 (seen at 00:07): 12 points
  - Reef 937 (seen at 00:07): 1 points
Processing: Qualification 26 - 2025 Iowa Regional.mp4...
  - Reef 2392 (seen at 00:05): 6 points
  - Reef 2393 (seen at 00:06): 0 points
Processing: Qualification 15 - 2025 Iowa Regional.mp4...
  - Reef 3378 (seen at 00:07): 0 points
  - Reef 3380 (seen at 00:08): 2 points
Processing: Qualification 43 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4...
  - Reef 4751 (seen at 00:05): 1 points
  - Reef 4752 (seen at 00:05): 10 points
Processing: Qualification 56 - 2025 Central Missouri Regional.mp4...
  - Reef 6533 (seen at 00:03): 9 points
  - Reef 6532 (seen at 00:03): 7 points
Processing: Final 1 - 2025 Central Missouri Regional.mp4...
  - Reef 7272 (seen at 00:03):